<a href="https://colab.research.google.com/github/kader-xai/data-science-roadmap/blob/main/module_47_helm_charts_for_ml.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

> ⛵ **Module 47 — Helm Charts for ML (templating + releases)** &nbsp;·&nbsp; part of [`data-science-roadmap`](https://github.com/kader-xai/data-science-roadmap)


# Module 47 — Helm Charts for ML

> M46 had Pods, Deployments, Services, PVCs — six YAML files for **one** vLLM deployment. Multiply by dev / staging / prod environments and a handful of models, and you're hand-editing dozens of files. **Helm** is the package manager for Kubernetes: bundle related YAML into a **chart**, parameterise it with **values**, install many copies as named **releases**.
>
> Almost every ML-native project ships its installation as a Helm chart: KServe, KubeRay, MLflow, Prometheus, NVIDIA GPU Operator. Knowing Helm is non-optional for production ML on k8s.

### What you'll cover
1. The 5-minute mental model — Chart, Values, Release
2. Anatomy of a chart — files and folders
3. **Templates** — Go templating + sprig functions
4. **values.yaml** — your chart's API
5. Building a real chart for a vLLM deployment (scaffolded)
6. **`helm install` / `upgrade` / `rollback` / `uninstall`**
7. **Dependencies** — composing charts (e.g. your app + Redis subchart)
8. **Hooks** — pre-install, post-upgrade, test
9. **Helmfile / Argo CD** — what teams use to manage 50 charts
10. ML-specific charts you'll meet in the wild


## 1 · 5-minute mental model

| Term | What it is | Analogy |
|---|---|---|
| **Chart** | a directory of templated YAML + a values schema | a package (`.deb`, `npm` module) |
| **Values** | parameters you supply at install time | the `--config` flags |
| **Release** | a *named installation* of a chart in a cluster | `apt install` invocation that you can `apt remove` |
| **Repository** | a server hosting many charts | apt repo, npm registry |

So: `helm install **release-name** **chart** -f values.yaml` — same chart, different values, multiple releases (one per env / per tenant).


## 2 · Anatomy of a chart

```
my-vllm/
├── Chart.yaml          # name, version, appVersion, dependencies
├── values.yaml         # default parameters
├── values.schema.json  # (optional) validate values at install time
├── templates/
│   ├── deployment.yaml # uses {{ .Values.image }}, {{ .Values.replicas }}, ...
│   ├── service.yaml
│   ├── ingress.yaml
│   ├── hpa.yaml
│   ├── configmap.yaml
│   ├── _helpers.tpl    # named template snippets
│   └── NOTES.txt       # printed after install — usage hints
├── charts/             # vendored sub-charts (e.g. redis)
└── tests/              # `helm test` connection / smoke tests
```

`Chart.yaml` is the metadata. `values.yaml` is the **public API** of your chart — the only thing users should need to read before installing.


## 3 · Templates — Go templating + sprig

Every file in `templates/` is rendered through Go's `text/template` plus the **sprig** function library.

In [ ]:
deployment_tpl = '''apiVersion: apps/v1
kind: Deployment
metadata:
  name: {{ include "my-vllm.fullname" . }}
  labels:
    {{- include "my-vllm.labels" . | nindent 4 }}
spec:
  replicas: {{ .Values.replicas }}
  selector:
    matchLabels:
      {{- include "my-vllm.selectorLabels" . | nindent 6 }}
  template:
    metadata:
      labels:
        {{- include "my-vllm.selectorLabels" . | nindent 8 }}
    spec:
      containers:
      - name: vllm
        image: "{{ .Values.image.repository }}:{{ .Values.image.tag }}"
        args:
        - --model
        - {{ .Values.model | quote }}
        - --max-model-len
        - {{ .Values.maxModelLen | quote }}
        ports:
        - containerPort: 8000
        {{- with .Values.resources }}
        resources:
          {{- toYaml . | nindent 10 }}
        {{- end }}
        envFrom:
        - secretRef:
            name: {{ include "my-vllm.fullname" . }}-hf-token
'''
print(deployment_tpl[:600], "...")

**Idioms in that template.**
- `.Values.X` — a parameter from `values.yaml`.
- `include "my-vllm.fullname" .` — call a named template defined in `_helpers.tpl`.
- `| nindent 4` — pipe through a function that adds a newline + indents 4 spaces. **The single most common Helm bug: wrong indentation.**
- `{{- ... -}}` — the dashes trim surrounding whitespace.
- `toYaml . | nindent N` — splat a values block (like `resources:`) into the template.
- `{{ .Values.x | quote }}` — wrap in quotes (matters for stringly-typed args).


## 4 · values.yaml — your chart's API

In [ ]:
values_yaml = '''replicas: 2

image:
  repository: vllm/vllm-openai
  tag: latest
  pullPolicy: IfNotPresent

model: Qwen/Qwen2.5-0.5B-Instruct
maxModelLen: 4096

resources:
  limits:
    nvidia.com/gpu: 1
    memory: 16Gi
    cpu: "4"
  requests:
    nvidia.com/gpu: 1
    memory: 12Gi
    cpu: "2"

service:
  type: ClusterIP
  port: 80

ingress:
  enabled: false
  className: nginx
  host: api.example.com

autoscaling:
  enabled: true
  minReplicas: 1
  maxReplicas: 8
  targetCPU: 70

hf:
  token: ""             # set via --set hf.token=$HF_TOKEN

extraEnv: []            # users can append env vars without forking
'''
print(values_yaml)

**Design rules for `values.yaml`.**
1. Group related fields (`image.*`, `service.*`, `ingress.*`).
2. Sensible defaults so `helm install foo my-vllm` *just works*.
3. **Add `enabled` toggles** for optional parts (`ingress.enabled`, `autoscaling.enabled`) — wrap their templates in `{{- if .Values.ingress.enabled }}`.
4. Always provide an `extraEnv` / `extraArgs` / `nodeSelector` / `tolerations` escape hatch so users don't have to fork the chart.

## 5 · Scaffold a chart in 1 minute

```bash
$ helm create my-vllm        # generates a working starter chart
$ tree my-vllm
my-vllm/
├── Chart.yaml
├── values.yaml
├── charts/
└── templates/
    ├── deployment.yaml
    ├── service.yaml
    ├── ingress.yaml
    ├── hpa.yaml
    ├── _helpers.tpl
    └── NOTES.txt
```

Then iterate. Render locally **before** deploying:

```bash
$ helm template my-vllm/ -f values-prod.yaml      # prints final YAML to stdout
$ helm lint my-vllm/                              # static checks
$ helm install --dry-run --debug my-rel my-vllm/  # render + validate against the cluster
```


## 6 · install / upgrade / rollback / uninstall

```bash
# install — release name "llm-prod", chart in ./my-vllm, values from prod.yaml
helm install llm-prod ./my-vllm -f values-prod.yaml --namespace llm --create-namespace

# upgrade with new values or new chart version
helm upgrade llm-prod ./my-vllm -f values-prod.yaml --set image.tag=v0.6.5

# what's installed?
helm list -n llm
helm history llm-prod -n llm

# rollback to a previous revision
helm rollback llm-prod 3 -n llm

# uninstall — removes all created objects
helm uninstall llm-prod -n llm
```

| Flag | What it does |
|---|---|
| `--set k=v` | override one value (CLI) |
| `-f file.yaml` | override many (file) — multiple `-f` flags merge |
| `--values` | alias of `-f` |
| `--dry-run` | render + send to API server, don't create |
| `--atomic` | rollback automatically if the upgrade fails |
| `--timeout 10m` | for slow LLM model pulls |
| `--wait` | wait until all resources are ready before returning |


## 7 · Dependencies — composing charts

In [ ]:
chart_yaml = '''apiVersion: v2
name: my-vllm
description: vLLM deployment with Redis cache
type: application
version: 0.1.0
appVersion: "0.6.4"
dependencies:
  - name: redis
    version: 19.0.0
    repository: https://charts.bitnami.com/bitnami
    condition: redis.enabled
'''
print(chart_yaml)

```bash
$ helm dependency update my-vllm/   # downloads sub-charts into charts/
$ helm install demo my-vllm/ \
    --set redis.enabled=true \
    --set redis.auth.password=secret123
```

Sub-charts let you reuse battle-tested components (Redis, Postgres, Kafka, MinIO) instead of re-templating them. Their values nest under their chart name (`redis.*` above).

## 8 · Hooks — pre-install, test

In [ ]:
hook_yaml = '''apiVersion: batch/v1
kind: Job
metadata:
  name: {{ include "my-vllm.fullname" . }}-warmup
  annotations:
    "helm.sh/hook": post-install,post-upgrade
    "helm.sh/hook-delete-policy": before-hook-creation,hook-succeeded
spec:
  template:
    spec:
      restartPolicy: Never
      containers:
      - name: warmup
        image: curlimages/curl:8.7.1
        command: ["sh","-c","until curl -sf http://{{ include "my-vllm.fullname" . }}/v1/models; do sleep 5; done"]
'''
print(hook_yaml)

**Common hook phases.** `pre-install`, `post-install`, `pre-upgrade`, `post-upgrade`, `pre-delete`, `post-delete`, `test`.

**`helm test`** — `templates/tests/*.yaml` annotated with `"helm.sh/hook": test` are run by `helm test <release>`. A typical ML smoke test: hit `/v1/models` and assert it returns the expected model.

## 9 · Helmfile / Argo CD — managing 50 charts

Once you have many charts, `helm install` from a laptop stops scaling. Two paths:

### Helmfile (script-style)
```yaml
# helmfile.yaml
releases:
  - name: llm-prod
    namespace: llm
    chart: ./charts/my-vllm
    values: [./values/prod.yaml]
  - name: prom
    namespace: monitoring
    chart: prometheus-community/kube-prometheus-stack
    values: [./values/prom.yaml]
```
`helmfile sync` applies all of them.

### Argo CD (GitOps-style — recommended)
- Argo CD watches a Git repo containing your chart + values.
- It **continuously reconciles** the cluster to match Git.
- Diffs, history, rollbacks, RBAC — all in a UI.
- This is what most production ML platforms use.

Either way: **never** run `helm install` from a developer laptop into prod. Releases live in Git, the cluster pulls from there.

## 10 · ML-specific charts you'll meet

| Chart | Installs |
|---|---|
| `kserve/kserve` | KServe ModelMesh + InferenceService CRDs |
| `kuberay/kuberay-operator` | Ray on Kubernetes |
| `kubeflow/training-operator` | PyTorch / TF / XGBoost distributed training CRDs |
| `nvidia/gpu-operator` | drivers + device plugin + DCGM exporter |
| `prometheus-community/kube-prometheus-stack` | Prometheus + Grafana + Alertmanager (M50) |
| `open-telemetry/opentelemetry-collector` | OTel collector (M51) |
| `bitnami/redis`, `bitnami/postgresql` | Redis (M37), Postgres (M36) |
| `qdrant/qdrant`, `weaviate/weaviate`, `milvus/milvus` | vector DBs (M42) |
| `argoproj/argo-cd` | the GitOps engine itself |
| `cert-manager/cert-manager` | TLS via Let's Encrypt |

A real ML platform is ~15-25 of these, composed into a single `argo-cd` app-of-apps.

### Anti-patterns
- ❌ Editing rendered manifests directly. Edit `values.yaml`, re-render.
- ❌ Putting secrets in `values.yaml` checked into Git. Use **Sealed Secrets**, External Secrets, or `helm-secrets` + SOPS.
- ❌ Pinning `appVersion` but leaving `image.tag: latest` in values.
- ❌ Forking a public chart instead of using its **values**. 99% of customisation is already exposed.
- ❌ One mega-chart that owns everything. Split by lifecycle (infra · platform · apps).


## ✅ Recap
- **Chart** = templates + values + metadata. **Release** = a named installation.
- `helm create` → edit `values.yaml` → render with `helm template` → `helm install`.
- Use **sub-charts** to compose (your app + Redis + Postgres).
- **Hooks** for warmup/migrations; `helm test` for smoke tests.
- Production: GitOps via **Argo CD**, not laptop `helm install`.
- The ML-native ecosystem ships almost entirely as Helm charts.

Next: **M48 — Terraform for ML Infrastructure** (the layer below Kubernetes).
